# Text generation with transformers from Hugging Face 🤗

## Setup
- activate environment: conda activate hugvenv312
- install requirements: pip install -r requirements.txt

In [34]:
# Check CPU, GPU, and ROCm installation
import platform  # For system/platform info
import torch     # For PyTorch and GPU checks
import subprocess  # For running shell commands

# Print platform and Python version
print(f'Platform: {platform.platform()}')
print(f'Python version: {platform.python_version()}')

# Print PyTorch version and check CUDA (NVIDIA GPU) availability
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'CUDA device name: {torch.cuda.get_device_name(0)}')
else:
    print('No CUDA GPU detected.')

# Try to print ROCm version (for AMD GPU support)
try:
    result = subprocess.run(['rocm-smi', '--showversion'], capture_output=True, text=True)
    print('ROCm SMI output:')
    print(result.stdout)
except Exception as e:
    print('ROCm SMI not found or not installed.')

Platform: Windows-11-10.0.26200-SP0
Python version: 3.12.13
PyTorch version: 2.9.1+rocm7.2.1
CUDA available: True
CUDA device count: 1
CUDA device name: AMD Radeon RX 7900 XT
ROCm SMI not found or not installed.


## Version 1 - Using the default pipeline as a high-level helper from HugginFace 🤗

In [35]:
from transformers import pipeline  # Import the pipeline helper

# Create a text-generation pipeline using the GPT-2 model from Hugging Face Hub
pipe = pipeline('text-generation', model='openai-community/gpt2')

# Define a prompt for the model to complete
prompt = 'What is machine learning?'
# Generate text using the pipeline
output = pipe(prompt)
# 'output' is a list of dicts with 'generated_text' key

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [36]:
# Display the generated text from the pipeline output
output[0]['generated_text']

"What is machine learning? It's going to take years to learn this type of information. This is why you need to wait until machine learning is ready before you hire one. However, after machine learning will tell the user that they're in a good"

## Version 2 - Using the model and tokenizer directly for more control

In [37]:
from transformers import AutoTokenizer, AutoModelForCausalLM  # Direct model/tokenizer usage

# Load the tokenizer and model directly for more control
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')

tokenizer # Display the tokenizer object to confirm it loaded correctly

GPT2TokenizerFast(name_or_path='openai-community/gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}

In [38]:
# Encode a sentence to input IDs (token IDs) for the model
sentence = 'What is machine learning?'
input_ids = tokenizer(sentence, return_tensors='pt')["input_ids"]
input_ids  # Tensor of token IDs

tensor([[2061,  318, 4572, 4673,   30]])

In [39]:
# Decode the input IDs back to text (should match the original sentence)
tokenizer.decode(input_ids[0])

'What is machine learning?'

### Tokenization breakdown for 'unbelievable'
This cell prints each token ID and its corresponding string for the word 'unbelievable'.
You can see how the tokenizer splits the word into subword tokens.


In [40]:
# Try encoding a different word to see its token IDs
sentence = 'unbelievable'
input_ids = tokenizer(sentence, return_tensors='pt').input_ids
input_ids  # Tensor of token IDs

tensor([[  403,  6667, 11203,   540]])

In [41]:
# Print each token ID and its corresponding decoded string for the word 'unbelievable'
for token_id in input_ids[0]:
    print(f'Token ID {token_id} is in the input {tokenizer.decode(token_id)}')

Token ID 403 is in the input un
Token ID 6667 is in the input bel
Token ID 11203 is in the input iev
Token ID 540 is in the input able


### Tokenization breakdown for 'homoscendasticity'
This cell demonstrates how the tokenizer handles a rare or complex word.
It prints each token ID and its decoded string, then decodes the full sequence back to text.
Notice how the tokenizer may split the word into multiple subword tokens if it is not in the vocabulary.

In [42]:
# Try a rare or complex word to see how the tokenizer splits it
word = 'homoscendasticity'
my_ids = tokenizer(word, return_tensors='pt').input_ids
my_ids  # Tensor of token IDs

# Print each token ID and its decoded string
for token_id in my_ids[0]:
    print(f'Token ID: {token_id} for input --> {tokenizer.decode(token_id)}')

# Decode the full sequence of token IDs back to text
tokenizer.decode(my_ids.squeeze())

Token ID: 26452 for input --> hom
Token ID: 418 for input --> os
Token ID: 15695 for input --> cend
Token ID: 3477 for input --> astic
Token ID: 414 for input --> ity


'homoscendasticity'